# SEC-RAG: final evaluation on vLLM (Kaggle, 2× T4)

Runs the held-out numeric eval (83 questions) with **Qwen2.5-7B-Instruct served by vLLM**, and measures how many questions per minute you get when requests run concurrently, compared with one at a time.

**Before you run anything:**
1. Upload `secrag-kaggle.zip` as a **private Kaggle Dataset** (it contains your code, index and XBRL facts; see the README).
2. In this notebook: **Add Input** → your dataset.
3. **Settings → Accelerator: GPU T4 ×2** and **Internet: On** (Internet needs a phone-verified Kaggle account).
4. Run the cells top to bottom. Total time is roughly 45–90 minutes.

In [1]:
import os, glob, shutil

DATASET_DIR = "/kaggle/input"          # searched recursively for your uploaded repo
MODEL = "Qwen/Qwen2.5-7B-Instruct"
WORK = "/kaggle/working/sec-rag"

# Keep both GPUs for vLLM. This notebook process (query embeddings, retrieval) runs on the CPU.
# Must be set BEFORE anything imports torch.
os.environ["CUDA_VISIBLE_DEVICES"] = ""

roots = [os.path.dirname(p) for p in glob.glob(f"{DATASET_DIR}/**/pyproject.toml", recursive=True)]
assert roots, "pyproject.toml not found: did you add your dataset as an input?"
print("repo found at:", roots[0])
if not os.path.exists(WORK):
    shutil.copytree(roots[0], WORK)    # /kaggle/input is read-only
for p in ["src/secrag", "data/index/chunks.jsonl", "data/index/embeddings.npy", "data/xbrl",
          "eval/queries_xbrl_holdout.jsonl"]:
    print(("OK      " if os.path.exists(f"{WORK}/{p}") else "MISSING ") + p)

repo found at: /kaggle/input/datasets/surajitpanja/rag-project-dataset
OK      src/secrag
OK      data/index/chunks.jsonl
OK      data/index/embeddings.npy
OK      data/xbrl
OK      eval/queries_xbrl_holdout.jsonl


In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


## 1. Install (about 5–10 minutes)
Ignore pip's dependency-conflict warnings about Kaggle's preinstalled packages.

In [3]:
!pip install -q vllm
!pip install -q -e "/kaggle/working/sec-rag[dense]"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 wh

## 2. Start the vLLM server
* T4s have no bfloat16, so the model runs in **fp16** (Ollama on the Mac used a 4-bit quantised build, so small accuracy differences are expected).
* `--tensor-parallel-size 2` splits the 15 GB of fp16 weights across both T4s.
* `--max-model-len 16384` matches the context window the Ollama fix used.

In [4]:
import subprocess, time, requests

LOG = "/kaggle/working/vllm.log"
cmd = ["python", "-m", "vllm.entrypoints.openai.api_server",
       "--model", MODEL, "--dtype", "half",
       "--tensor-parallel-size", "2",
       "--max-model-len", "16384",
       "--gpu-memory-utilization", "0.90",
       "--max-num-seqs", "16",
       "--port", "8000"]
server = subprocess.Popen(cmd, env=dict(os.environ, CUDA_VISIBLE_DEVICES="0,1"),
                          stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

def tail(n=40):
    print("".join(open(LOG).readlines()[-n:]))

for i in range(120):                     # model download + load: usually 3-8 minutes
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).ok:
            print(f"vLLM ready after ~{i * 10}s")
            break
    except requests.RequestException:
        pass
    if server.poll() is not None:
        tail(80)
        raise RuntimeError("vLLM exited: see the log above")
    time.sleep(10)
else:
    tail(80)
    raise RuntimeError("vLLM did not become ready in 20 minutes")

vLLM ready after ~270s


**If the server fails to start:**
* *Out of memory during CUDA-graph capture:* add `"--enforce-eager"` to `cmd` and re-run the cell.
* *An attention-backend or compute-capability error:* your vLLM version may not support T4 any more. Copy the last ~50 log lines (`tail(50)`) and fix the version before going on.
* To restart: `server.kill()`, then re-run the cell.

## 3. Point secrag at vLLM and sanity-check one question

In [5]:
os.environ.update({
    "LLM_BASE_URL": "http://localhost:8000/v1",
    "LLM_API_KEY": "EMPTY",
    "LLM_MODEL": MODEL,
    "SECRAG_DATA_DIR": f"{WORK}/data",
})
os.chdir(WORK)
!python -m secrag ask "What was Apple's net income for fiscal 2026?" --trace

config_sentence_transformers.json: 100%|████████| 124/124 [00:00<00:00, 573kB/s]
README.md: 100%|███████████████████████████| 94.8k/94.8k [00:00<00:00, 59.7MB/s]
sentence_bert_config.json: 100%|██████████████| 52.0/52.0 [00:00<00:00, 267kB/s]
config.json: 100%|█████████████████████████████| 743/743 [00:00<00:00, 3.70MB/s]

model.safetensors: downloading bytes: ██████████████▎       | 86.7MB, 7.64MB/s  
model.safetensors: downloading bytes: ██████████████████████| 86.7MB, 8.32MB/s  
model.safetensors: reconstructing file: 100%|██████|  133MB /  133MB, 13.0MB/s  
tokenizer_config.json: 100%|███████████████████| 366/366 [00:00<00:00, 1.65MB/s]
vocab.txt: 100%|█████████████████████████████| 232k/232k [00:00<00:00, 20.9MB/s]
tokenizer.json: 100%|████████████████████████| 711k/711k [00:00<00:00, 51.1MB/s]
config.json: 100%|██████████████████████████████| 190/190 [00:00<00:00, 536kB/s]

Apple's net income for fiscal 2026 was $132,175 million. [1][2][3]

  [1] Apple Inc. 10-K (period ending 20

## 4. Single-stream latency (10 questions, one at a time)
This is the same setup as the Mac/Ollama runs, so the latencies can be compared directly.

In [6]:
!python -m secrag eval-qa --queries eval/queries_xbrl_holdout.jsonl --configs no_verifier --no-judge --limit 10 --workers 1

Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 2958.13it/s]
  [no_verifier] 1/10 xbrl-AAPL-2025-r-d-expense: correct=True claims=1 gold_in_evidence=True judged_ok=None
  [no_verifier] 2/10 xbrl-AAPL-2025-eps-diluted: correct=True claims=1 gold_in_evidence=True judged_ok=None
  [no_verifier] 3/10 xbrl-AAPL-2025-operating-cash-flow: correct=True claims=1 gold_in_evidence=True judged_ok=None
  [no_verifier] 4/10 xbrl-AAPL-2025-total-assets: correct=True claims=1 gold_in_evidence=True judged_ok=None
  [no_verifier] 5/10 xbrl-AAPL-2026-revenue: correct=True claims=1 gold_in_evidence=True judged_ok=None
  [no_verifier] 6/10 xbrl-AAPL-2026-net-income: correct=True claims=1 gold_in_evidence=True judged_ok=None
  [no_verifier] 7/10 xbrl-AAPL-2026-operating-income: correct=True claims=1 gold_in_evidence=True judged_ok=None
  [no_verifier] 8/10 xbrl-AAPL-2026-eps-diluted: correct=True claims=1 gold_in_evidence=True judged_ok=None
  [no_verifier] 9/10 xbrl-AAPL-2026-total-asse

## 5. Final held-out eval: 83 questions × 2 configs, 16 concurrent requests
vLLM batches the concurrent requests on the GPU, so this is where throughput shows up.
The judge is off (`--no-judge`): the numeric accuracy metrics don't need it.

In [7]:
!python -m secrag eval-qa --queries eval/queries_xbrl_holdout.jsonl --configs no_verifier verifier_no_xbrl --no-judge --workers 16

Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 2911.05it/s]
  [no_verifier] 1/83 xbrl-AAPL-2025-total-assets: correct=True claims=1 gold_in_evidence=True judged_ok=None
  [no_verifier] 2/83 xbrl-AAPL-2026-total-assets: correct=True claims=1 gold_in_evidence=True judged_ok=None
  [no_verifier] 3/83 xbrl-AAPL-2026-net-income: correct=True claims=1 gold_in_evidence=True judged_ok=None
  [no_verifier] 4/83 xbrl-AMZN-2024-total-assets: correct=True claims=1 gold_in_evidence=True judged_ok=None
  [no_verifier] 6/83 xbrl-AMZN-2024-net-income: correct=True claims=1 gold_in_evidence=True judged_ok=None
  [no_verifier] 5/83 xbrl-AAPL-2025-r-d-expense: correct=True claims=1 gold_in_evidence=True judged_ok=None
  [no_verifier] 7/83 xbrl-AMZN-2024-operating-income: correct=True claims=1 gold_in_evidence=True judged_ok=None
  [no_verifier] 8/83 xbrl-AMZN-2024-eps-diluted: correct=True claims=1 gold_in_evidence=True judged_ok=None
  [no_verifier] 9/83 xbrl-AAPL-2025-operating-cas

## 6. Summary and comparison with the Mac (Ollama) runs

In [8]:
import json

runs = sorted(glob.glob("eval/results/qa-*.json"))
loaded = [(f, json.load(open(f))["summary"]) for f in runs]
# latest single-stream run (step 4) and latest concurrent run (step 5), picked by content, not position
single_file, single = [(f, s["no_verifier"]) for f, s in loaded
                       if s.get("no_verifier", {}).get("workers") == 1][-1]
final_file, final = [(f, s) for f, s in loaded if any(r.get("workers", 1) > 1 for r in s.values())][-1]

# Measured earlier on a MacBook Air with Ollama (qwen2.5:7b, 4-bit), sequential, same 83 questions:
OLLAMA = {"no_verifier":      {"accuracy": 0.904, "p50": 26.5},
          "verifier_no_xbrl": {"accuracy": 0.904, "p50": 31.6}}

def pct(x): return "-" if x is None else f"{100 * x:.1f}%"

print(f"Single-stream p50 latency on vLLM: {single['p50_latency_s']:.1f}s per question "
      f"(Ollama on Mac: {OLLAMA['no_verifier']['p50']}s)\n")
print("| config | accuracy (vLLM) | accuracy (Ollama) | precision when answering | confident errors | questions/min (vLLM, 16 workers) | approx questions/min (Ollama, sequential) |")
print("|---|---|---|---|---|---|---|")
for name, r in final.items():
    o = OLLAMA.get(name, {})
    o_qpm = f"{60 / o['p50']:.1f}" if o else "-"
    print(f"| {name} | {pct(r['numeric_accuracy'])} | {pct(o.get('accuracy'))} | {pct(r['precision_when_answering'])} | "
          f"{pct(r['confident_errors'])} | {r['questions_per_min']} | {o_qpm} |")

Single-stream p50 latency on vLLM: 7.3s per question (Ollama on Mac: 26.5s)

| config | accuracy (vLLM) | accuracy (Ollama) | precision when answering | confident errors | questions/min (vLLM, 16 workers) | approx questions/min (Ollama, sequential) |
|---|---|---|---|---|---|---|
| no_verifier | 89.2% | 90.4% | 89.2% | 10.8% | 15.61 | 2.3 |
| verifier_no_xbrl | 89.2% | 90.4% | 91.4% | 8.4% | 11.25 | 1.9 |


## 7. Save outputs
Download `secrag_vllm_outputs.zip` from the **Output** panel on the right, and put the JSON files in `eval/results/` in your repo.

In [9]:
out = "/kaggle/working/outputs"
os.makedirs(out, exist_ok=True)
for f in [single_file, final_file, LOG]:
    shutil.copy(f, out)
shutil.make_archive("/kaggle/working/secrag_vllm_outputs", "zip", out)
print(os.listdir(out))
server.kill()

['vllm.log', 'qa-20261117-065149.json', 'qa-20261117-070435.json']
